In [2]:
import os
import json
from openai import OpenAI
import requests

client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=os.environ["GITHUB_TOKEN"]
)

In [14]:
# Las herramientas que el agente puede usar
def obtener_tasa_dolar():
    """Simula consultar la TRM actual"""
    return {"tasa": 3710.5, "fecha": "2026-03-15", "fuente": "Banco de la República"}

def calcular_conversion(monto_cop, tasa):
    """Convierte pesos colombianos a dólares"""
    return {"monto_usd": round(monto_cop / tasa, 2), "monto_cop": monto_cop, "tasa_usada": tasa}

def obtener_tasa_dolar_real():
    """Consulta la TRM real desde la API del Banco de la República"""
    try:
        url = 'https://www.datos.gov.co/resource/32sa-8pi3.json?$limit=1&$order=vigenciadesde DESC'
        respuesta = requests.get(url, timeout=5)
        dato = respuesta.json()[0]
        return {
            'tasa': float(dato['valor']),
            'fecha': dato['vigenciadesde'][:10],
            'fuente': 'Banco de la República Colombia'
        }
    except Exception as e:
        return {'error': str(e)}
    
def obtener_inflacion_colombia():
    """Retorna la inflación proyectada de Colombia según el Plan Financiero 2026"""
    return {
        "inflacion_proyectada": 5.1,
        "meta_banrep": 3.0,
        "año": 2026,
        "fuente": "Ministerio de Hacienda - Plan Financiero 2026"
    }

In [18]:
# Definición de herramientas para el modelo
herramientas = [
    {
        "type": "function",
        "function": {
            "name": "obtener_tasa_dolar",
            "description": "Obtiene la tasa de cambio actual del dólar en Colombia (TRM)",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "calcular_conversion",
            "description": "Convierte un monto en pesos colombianos a dólares",
            "parameters": {
                "type": "object",
                "properties": {
                    "monto_cop": {"type": "number", "description": "Monto en pesos colombianos"},
                    "tasa": {"type": "number", "description": "Tasa de cambio a usar"}
                },
                "required": ["monto_cop", "tasa"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "obtener_inflacion_colombia",
            "description": "Obtiene la inflación proyectada de Colombia y la meta del Banco de la República",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    }
]

In [22]:
def ejecutar_herramienta(nombre, argumentos):
    if nombre == "obtener_tasa_dolar":
        return obtener_tasa_dolar_real()
    elif nombre == "calcular_conversion":
        return calcular_conversion(**argumentos)
    elif nombre == "obtener_inflacion_colombia":
        return obtener_inflacion_colombia()

def agente(pregunta):
    mensajes = [{"role": "user", "content": pregunta}]
    
    while True:
        respuesta = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=mensajes,
            tools=herramientas
        )
        
        mensaje = respuesta.choices[0].message
        
        # Si no hay tool calls, el agente terminó
        if not mensaje.tool_calls:
            print(mensajes)
            return mensaje.content
        
        # Ejecutar cada herramienta que pidió el modelo
        mensajes.append(mensaje)
        for tool_call in mensaje.tool_calls:
            nombre = tool_call.function.name
            argumentos = json.loads(tool_call.function.arguments)
            resultado = ejecutar_herramienta(nombre, argumentos)
            
            print(f"  → Herramienta usada: {nombre}({argumentos})")
            
            mensajes.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(resultado)
            })

In [7]:
# Prueba
print("Pregunta: ¿Cuántos dólares son 500,000 pesos?")
respuesta = agente("¿Cuántos dólares son 500,000 pesos colombianos?")
print(f"Respuesta: {respuesta}\n")

Pregunta: ¿Cuántos dólares son 500,000 pesos?
  → Herramienta usada: obtener_tasa_dolar({})
  → Herramienta usada: calcular_conversion({'monto_cop': 500000, 'tasa': 3660.1})
[{'role': 'user', 'content': '¿Cuántos dólares son 500,000 pesos colombianos?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_N1E9d5ZXF0a3YSJhoAp6giHM', function=Function(arguments='{}', name='obtener_tasa_dolar'), type='function')]), {'role': 'tool', 'tool_call_id': 'call_N1E9d5ZXF0a3YSJhoAp6giHM', 'content': '{"tasa": 3660.1, "fecha": "2026-04-01", "fuente": "Banco de la Rep\\u00fablica Colombia"}'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_npT3ownfNflTZhbLJ04HQ2oS', function=Function(arguments='{"monto_cop":500000,"tasa":3660.1}', name='calcular_convers

In [23]:
print(agente("¿Cuántos dólares son 2,000,000 de pesos?"))

  → Herramienta usada: obtener_tasa_dolar({})
  → Herramienta usada: calcular_conversion({'monto_cop': 2000000, 'tasa': 3660.1})
[{'role': 'user', 'content': '¿Cuántos dólares son 2,000,000 de pesos?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_4ojQGmyZFkwwb23Tsi3oRdDO', function=Function(arguments='{}', name='obtener_tasa_dolar'), type='function')]), {'role': 'tool', 'tool_call_id': 'call_4ojQGmyZFkwwb23Tsi3oRdDO', 'content': '{"tasa": 3660.1, "fecha": "2026-04-01", "fuente": "Banco de la Rep\\u00fablica Colombia"}'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_k6wHA9IZRJvjoXPTrY3KatyE', function=Function(arguments='{"monto_cop":2000000,"tasa":3660.1}', name='calcular_conversion'), type='function')]), {'role': 'tool', 'tool_c

In [24]:
print(agente("¿Cómo está la inflación en Colombia?"))

  → Herramienta usada: obtener_inflacion_colombia({})
[{'role': 'user', 'content': '¿Cómo está la inflación en Colombia?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_CfQcScOKDusW6R0FHjSZwyeD', function=Function(arguments='{}', name='obtener_inflacion_colombia'), type='function')]), {'role': 'tool', 'tool_call_id': 'call_CfQcScOKDusW6R0FHjSZwyeD', 'content': '{"inflacion_proyectada": 5.1, "meta_banrep": 3.0, "a\\u00f1o": 2026, "fuente": "Ministerio de Hacienda - Plan Financiero 2026"}'}]
La inflación proyectada en Colombia es del 5.1% para el año 2026, mientras que la meta del Banco de la República (Banrep) es del 3.0%. Esta información proviene del Ministerio de Hacienda en el Plan Financiero 2026.


In [25]:
print(agente("Con la inflación actual, ¿conviene guardar los ahorros en dólares?"))

  → Herramienta usada: obtener_inflacion_colombia({})
  → Herramienta usada: obtener_tasa_dolar({})
[{'role': 'user', 'content': 'Con la inflación actual, ¿conviene guardar los ahorros en dólares?'}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_64BSxlvhEbjbec7tbiFJO0DV', function=Function(arguments='{}', name='obtener_inflacion_colombia'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_m5bVvlQGYVSJkJxrj5tyVRc0', function=Function(arguments='{}', name='obtener_tasa_dolar'), type='function')]), {'role': 'tool', 'tool_call_id': 'call_64BSxlvhEbjbec7tbiFJO0DV', 'content': '{"inflacion_proyectada": 5.1, "meta_banrep": 3.0, "a\\u00f1o": 2026, "fuente": "Ministerio de Hacienda - Plan Financiero 2026"}'}, {'role': 'tool', 'tool_call_id': 'call_m5bVvlQGYVSJkJxrj5tyVRc0', 'content': '{"tasa": 3660.1, "fecha": "2026-04-01", "fuente": "Banco de la Rep\